In [3]:
import pandas as pd
import numpy as np
import glob
import os

# 1. Locate all CSV files inside the data directory
csv_files = glob.glob("data/*.csv")
print(f"Found files: {csv_files}")

# 2. Load each file and tag the snapshot source
df_list = []
for file_path in csv_files:
    temp_df = pd.read_csv(file_path, low_memory=False)
    # Extract filename as source indicator
    temp_df["_source_file"] = os.path.basename(file_path)
    df_list.append(temp_df)

# 3. Concatenate all datasets into a single master DataFrame
raw_df = pd.concat(df_list, ignore_index=True)
print(f"Combined DataFrame Shape: {raw_df.shape}")

Found files: ['data\\albany_listings_february.csv', 'data\\Albany_listings_january.csv', 'data\\Albany_listings_june.csv']
Combined DataFrame Shape: (1377, 22)


In [4]:
raw_df.head(10)


,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,...,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,_snapshot_month,_ingested_at,_source_file
0,2992450,Luxury 2 bedroom apartment,4621559,1462661762635155041,Kenneth,NaN,THIRD WARD,42.65789,-73.75370,Entire home/apt,...,9,2022-08-17,0.06,1,321,0,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv
1,3820211,Restored Precinct in Center Sq. w/Parking,19648678,1463087024016718788,Ming,NaN,SIXTH WARD,42.65222,-73.76724,Entire home/apt,...,317,2026-01-25,2.26,7,301,10,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv
2,5651579,Large studio apt by Capital Center & ESP@,29288920,1465431505328745177,Gregg,NaN,SECOND WARD,42.64615,-73.75966,Entire home/apt,...,389,2026-02-06,2.96,2,35,20,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv
3,6015313,Comfortable 3BR Home w/ Central Air | Pine Hills,31223807,1465459398999270556,Ryan,NaN,THIRTEENTH WARD,42.67198,-73.79339,Entire home/apt,...,0,NaN,NaN,1,324,0,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv
4,6623339,Center Sq. Loft in Converted Precinct w/ Parking,19648678,1463087024016718788,Ming,NaN,SIXTH WARD,42.65222,-73.76724,Entire home/apt,...,332,2025-06-30,2.55,7,348,1,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv
5,9005989,"Studio in The heart of Center SQ, in Albany NY",17766924,1463028469002798404,Sugey,NaN,SIXTH WARD,42.65559,-73.76506,Entire home/apt,...,623,2025-08-03,5.37,1,16,13,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv
6,9501054,Spacious suite with full bath by Capital Center,29288920,1465431505328745177,Gregg,NaN,SECOND WARD,42.64734,-73.75851,Private room,...,448,2026-01-25,3.67,2,252,38,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv
7,10768745,Alb hospital area studio bath wifi. (Red),5691268,1462687000374947891,Elizabeth,NaN,FOURTEENTH WARD,42.66078,-73.80553,Private room,...,871,2026-01-22,7.13,2,183,111,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv
8,11253948,/Fire Place Bungalow\ 1917 SUNY Eagle 6Beds 2B...,4259750,1462534529294832741,Peter,NaN,FIFTEENTH WARD,42.67567,-73.82438,Entire home/apt,...,220,2026-02-11,1.86,8,362,43,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv
9,11639446,$55twin($30 foreign student)FreeBF Noa/c no smoke,61700428,1466759141736283314,Abdullah,NaN,ELEVENTH WARD,42.67583,-73.78297,Private room,...,231,2024-02-29,1.95,2,364,0,NaN,february,2026-07-22T12:37:59.682431+00:00,albany_listings_february.csv


In [8]:
print(raw_df.columns)

Index(['id', 'name', 'host_id', 'host_profile_id', 'host_name',
       'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude',
       'room_type', 'price', 'minimum_nights', 'number_of_reviews',
       'last_review', 'reviews_per_month', 'calculated_host_listings_count',
       'availability_365', 'number_of_reviews_ltm', 'license',
       '_snapshot_month', '_ingested_at', '_source_file'],
      dtype='str')


In [4]:
df = raw_df.copy()

# 1. Clean Column Names (lowercase, strip, replace spaces/special chars)
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_")
              .str.replace(r"[^\w\s]", "", regex=True)
)

# 2. Clean Currency Fields ($1,250.00 -> 1250.0)
currency_cols = ["price", "weekly_price", "monthly_price", "extra_people", "security_deposit", "cleaning_fee"]
for col in currency_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(r"[\$,]", "", regex=True)
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 3. Standardise Phone Numbers (if host_phone or phone column exists)
phone_cols = [c for c in df.columns if "phone" in c]
for col in phone_cols:
    df[col] = df[col].astype(str).str.replace(r"\D", "", regex=True)

# 4. Standardise Dates & Data Types
date_cols = [c for c in df.columns if "date" in c or "since" in c]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_cols = ["bedrooms", "bathrooms", "beds", "accommodates", "availability_365", "number_of_reviews"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 5. Remove Duplicates
before_dup = len(df)
df = df.drop_duplicates(subset=["id", "_snapshot_month"], keep="first")
print(f"Removed {before_dup - len(df)} duplicates.")

# 6. Feature Engineering: Estimated Monthly Revenue
# Est Revenue = Price * (30 - (availability_365 / 12)) or Price * Reviews * Minimum Nights
df["est_monthly_revenue"] = df["price"] * (30 - (df["availability_365"].fillna(365) / 12).clip(0, 30))

print("Data Cleaning Complete!")

Removed 0 duplicates.
Data Cleaning Complete!


In [10]:
# Question 1: Top Neighbourhoods by Estimated Monthly Revenue
q1 = (
    df.groupby('neighbourhood_clean')
    .agg(
        avg_monthly_revenue=('est_monthly_revenue', 'mean'),
        median_nightly_price=('price', 'median'),
        avg_occupancy_rate=('est_occupancy_rate', 'mean'),
        total_listings=('id', 'count')
    )
    .sort_values(by='avg_monthly_revenue', ascending=False)
    .head(10)
)
q1

NameError: name 'df' is not defined

In [11]:
# At the end of your cleaning cell, make sure you have:
df = raw_df.copy()

# ... (your cleaning logic and feature engineering here) ...

# Ensure 'df' exists at the end of the cell!
print("Data cleaning complete. 'df' is ready!")

Data cleaning complete. 'df' is ready!


In [12]:
# Question 2: Room Type Performance Analysis
q2 = (
    df.groupby('room_type')
    .agg(
        avg_nightly_price=('price', 'mean'),
        median_nightly_price=('price', 'median'),
        avg_occupancy_rate=('est_occupancy_rate', 'mean'),
        avg_monthly_revenue=('est_monthly_revenue', 'mean'),
        total_listings=('id', 'count')
    )
    .sort_values(by='avg_monthly_revenue', ascending=False)
)
q2

KeyError: "Label(s) ['est_monthly_revenue', 'est_occupancy_rate'] do not exist"